# Baseline Model — Classical Bead Detection + MLflow Tracking

Before touching a U-Net, we need a baseline: a simple, self-contained method scored on your
real annotated data. Every future model (including the deep one) has to actually beat this
number, measured the same way — not just "seem better."

**What is MLflow?** An experiment tracker. Every time you try a model or a different setting,
it logs the parameters you used and the metrics you got, as one "run." Without this, it's easy
to lose track of which threshold value produced which result once you've tried a dozen
variations — MLflow keeps that history automatically and gives you a UI to compare runs
side by side.

## Step 0: Imports

The `sys.path` lines below are a safety net: if `axonbead_ml` isn't importable as an installed
package on your machine yet, this adds`src/` directly to Python's search path so the notebook
works regardless. Worth actually fixing the underlying install at some point, but this keeps you 
unblocked for now.

In [18]:
from pathlib import Path

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
from bioio import BioImage

from axonbead_ml.data.loading import load_annotations_with_condition, list_all_images_with_condition
from axonbead_ml.models.baseline import detect_beads
from axonbead_ml.training.evaluate import match_points
print("All imports done!")

All imports done!


## Step 1: Load annotations and image list

Same loading logic as the EDA notebook — now imported from `src/` instead of copy-pasted,
since this is the second notebook that needs it.

In [2]:
raw_dir = Path("../data/raw")
annotations_path = Path("../data/interim/all_annotations.csv")

annotations = load_annotations_with_condition(annotations_path, raw_dir)
images = list_all_images_with_condition(raw_dir)
print(f"{len(images)} images, {len(annotations)} total annotated beads")

60 images, 766 total annotated beads


## Step 2: Run the baseline detector on every image

For each image: load it, run the classical detector, compare its predicted points against
your annotated ground truth using `match_points`. This is the same evaluation code the U-Net
will use later, so results are directly comparable.

**Baseline parameters below are a first guess, not tuned** — `min_area`, `max_area`, and
`min_circularity` come from typical bead sizes, but you may need to adjust them after seeing
the results. That's expected and fine; log each attempt as its own MLflow run (Step 4) so you
can compare.

In [14]:
THRESHOLD_METHOD = "manual"
MANUAL_THRESHOLD = 225
MIN_AREA = 25
MAX_AREA = 200
MIN_CIRCULARITY = 0.3
MAX_MATCH_DISTANCE = 15.0  # pixels

results = []
for _, row in images.iterrows():
    img = BioImage(str(row["path"]))
    image = img.get_image_data("YX", C=1, T=0, Z=0)

    predicted = detect_beads(
        image,
        threshold_method=THRESHOLD_METHOD,
        manual_threshold=MANUAL_THRESHOLD,
        min_area=MIN_AREA,
        max_area=MAX_AREA,
        min_circularity=MIN_CIRCULARITY,
    )
    ground_truth = annotations.loc[annotations["image"] == row["image"], ["y", "x"]].values

    metrics = match_points(predicted, ground_truth, max_distance=MAX_MATCH_DISTANCE)
    metrics["image"] = row["image"]
    metrics["condition"] = row["condition"]
    metrics["n_predicted"] = len(predicted)
    metrics["n_true"] = len(ground_truth)
    results.append(metrics)

results_df = pd.DataFrame(results)
results_df.head()

,true_positives,false_positives,false_negatives,precision,recall,f1,image,condition,n_predicted,n_true
0,2,32,1,0.058824,0.666667,0.108108,NI231117_SMI31-488_20x_HCl_01.czi,control,34,3
1,2,15,1,0.117647,0.666667,0.200000,NI231117_SMI31-488_20x_HCl_02.czi,control,17,3
2,1,16,0,0.058824,1.000000,0.111111,NI231117_SMI31-488_20x_HCl_03.czi,control,17,1
3,0,28,1,0.000000,0.000000,0.000000,NI231117_SMI31-488_20x_HCl_04.czi,control,28,1
4,1,16,1,0.058824,0.500000,0.105263,NI231117_SMI31-488_20x_HCl_05.czi,control,17,2


## Step 3: Aggregate metrics

Overall performance, and broken down by condition — a model can look fine on average while
quietly failing on one condition, so check both.

In [15]:
overall_tp = results_df["true_positives"].sum()
overall_fp = results_df["false_positives"].sum()
overall_fn = results_df["false_negatives"].sum()
overall_precision = overall_tp / (overall_tp + overall_fp) if (overall_tp + overall_fp) else 0.0
overall_recall = overall_tp / (overall_tp + overall_fn) if (overall_tp + overall_fn) else 0.0
overall_f1 = (
    2 * overall_precision * overall_recall / (overall_precision + overall_recall)
    if (overall_precision + overall_recall) else 0.0
)

print(f"Overall — precision: {overall_precision:.3f}, recall: {overall_recall:.3f}, f1: {overall_f1:.3f}")
print()
print("By condition:")
results_df.groupby("condition")[["precision", "recall", "f1"]].mean().round(3)

Overall — precision: 0.237, recall: 0.607, f1: 0.341

By condition:


,precision,recall,f1
condition,,,
control,0.133,0.666,0.165
high_beads,0.364,0.648,0.455
low_beads,0.278,0.473,0.316


**Takeaway:** recall is much higher than precision across the board, especially in control (0.666 recall vs. 0.133 precision). That means the manual threshold at 220 is catching real beads reasonably well, but it's also flagging a lot of non-bead structures — likely background noise or neurite segments that happen to cross that brightness threshold but don't get filtered out by your current min_area/max_area/min_circularity settings.

This highlights the limitations of classical filtering, which leaves a lot of false positives on the table. The deep learning model has a real chance to fix by learning appearance, not just brightness+shape.

## Step 4: Log this run to MLflow

This is what makes the result reusable later — the parameters and metrics get saved together,
so when you come back in Week 4/5 with a trained U-Net, you can look up exactly what this
baseline scored rather than re-running it from memory.

In [22]:
# Tell mlflow to use the mlruns folder in the project root:
db_path = Path("../mlflow.db").resolve().as_posix()
mlflow.set_tracking_uri(f"sqlite:///{db_path}")

mlflow.set_experiment("axonbead-ml")

with mlflow.start_run(run_name="baseline_classical_threshold"):
    mlflow.log_param("model_type", "classical_threshold")
    mlflow.log_param("threshold_method", THRESHOLD_METHOD)
    mlflow.log_param("min_area", MIN_AREA)
    mlflow.log_param("max_area", MAX_AREA)
    mlflow.log_param("min_circularity", MIN_CIRCULARITY)
    mlflow.log_param("max_match_distance", MAX_MATCH_DISTANCE)

    mlflow.log_metric("precision", overall_precision)
    mlflow.log_metric("recall", overall_recall)
    mlflow.log_metric("f1", overall_f1)

    for condition, group in results_df.groupby("condition"):
        mlflow.log_metric(f"f1_{condition}", group["f1"].mean())

    results_path = Path("../docs/experiments/baseline_results.csv")
    results_df.to_csv(results_path, index=False)
    mlflow.log_artifact(str(results_path))

    print(f"Run logged. Run ID: {mlflow.active_run().info.run_id}")

Run logged. Run ID: e44d45c8a6f6400997b6b7e2f6fa2d16


## Step 5: View results in the MLflow UI

From a terminal, in the project root (not inside the notebook):

```
mlflow ui
```

Then open http://localhost:5000 in your browser. You'll see this run listed with all its
parameters and metrics — this is also where every future run (different threshold settings,
and eventually the U-Net) will show up side by side for comparison.

## Next

Try adjusting `THRESHOLD_METHOD`, `MIN_AREA`/`MAX_AREA`, or `MIN_CIRCULARITY` in Step 2 and
re-running Steps 2–4 — each run gets its own entry in MLflow, so nothing gets overwritten.
Once you're happy with a baseline number, that becomes the target the U-Net needs to beat.